# All Models Guide: Current Model Names
# 全模型指南：当前模型名称

PipelineTS model names used by `ModelPipeline` and `SmartRouter` are registry keys, not class names. This notebook documents the latest names and shows how to choose a practical candidate set for industrial workloads.

PipelineTS 在 `ModelPipeline` 和 `SmartRouter` 中使用的是模型注册名，而不是类名。本教程展示最新名称，并说明如何为工业工作负载选择候选模型。

In [ ]:
from PipelineTS.pipeline import ModelPipeline

available = ModelPipeline.list_all_available_models()
print(f"{len(available)} models are available in this environment:")
for name in available:
    print("-", name)

## Current core names / 当前核心名称

| Family / 家族 | Model names / 模型名 | Typical use / 典型场景 |
|---|---|---|
| Statistical / 统计 | `auto_arima`, `prophet` | Strong seasonality, explainable baselines |
| Native ML / 原生机器学习 | `catboost`, `xgboost`, `random_forest`, `extra_forest`, `gc_forest`, `wide_gbrt` | Retail demand, operations metrics, tabular covariates |
| Sklearn wrappers / sklearn 封装 | `multi_output_model`, `multi_step_model`, `regressor_chain` | Fast baselines and robust production challengers |
| NN light / 轻量 NN | `d_linear`, `n_linear`, `tide`, `tcn` | Fast neural baselines, trend/seasonality patterns |
| NN medium/heavy / 中大型 NN | `n_beats`, `n_hits`, `tft`, `gau`, `stacking_rnn`, `time2vec`, `transformer`, `patch_rnn`, `deepar` | Complex patterns when enough history exists |
| Multivariate NN / 多变量 NN | `itransformer`, `srs_net` | Multi-input or multi-output industrial sensor/load forecasting |
| Optional foundation models / 可选基础模型 | `chronos_2`, `chronos_2_synth`, `chronos_2_small` | Zero-shot forecasting when `chronos-forecasting` is installed |

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

def make_retail_demand(n_days=240, n_stores=1, start="2023-01-01"):
    rows = []
    for i in range(n_stores):
        rng = np.random.default_rng(100 + i)
        dates = pd.date_range(start, periods=n_days, freq="D")
        dow = dates.dayofweek.to_numpy()
        month = dates.month.to_numpy()
        holiday = ((dow >= 5) | rng.binomial(1, 0.04, n_days).astype(bool)).astype(int)
        promotion = rng.binomial(1, 0.14 + 0.06 * (dow >= 4), n_days).astype(int)
        price_index = 1.0 + 0.04 * np.sin(np.linspace(0, 5 * np.pi, n_days)) + rng.normal(0, 0.015, n_days)
        temperature = 18 + 10 * np.sin(np.linspace(-0.8, 2.8 * np.pi, n_days)) + rng.normal(0, 1.8, n_days)
        stockout = rng.binomial(1, 0.025, n_days)
        baseline = 120 + 18 * i
        weekly = np.where(dow < 5, 8, 28)
        seasonal = 16 * np.sin(2 * np.pi * np.arange(n_days) / 365.25 + i / 3)
        trend = 0.08 * np.arange(n_days)
        demand = (
            baseline + weekly + seasonal + trend
            + 34 * promotion + 22 * holiday
            + 0.9 * np.maximum(temperature - 20, 0)
            - 75 * (price_index - 1.0)
            - 45 * stockout
            + rng.normal(0, 7, n_days)
        )
        rows.append(pd.DataFrame({
            "date": dates,
            "store_id": f"store_{i + 1:02d}",
            "sales": np.maximum(demand, 1),
            "promotion": promotion,
            "holiday": holiday,
            "price_index": price_index,
            "temperature": temperature,
            "stockout": stockout,
            "month": month,
        }))
    return pd.concat(rows, ignore_index=True)

In [ ]:
data = make_retail_demand(n_days=180, n_stores=1).drop(columns=["store_id"])
train, valid = data.iloc[:-12].copy(), data.iloc[-12:].copy()
safe_models = [m for m in ["random_forest", "extra_forest", "multi_output_model", "multi_step_model"] if m in available]
safe_models

In [ ]:
from PipelineTS.pipeline import ModelPipeline

pipe = ModelPipeline(
    time_col="date",
    target_col="sales",
    lags=12,
    include_models=safe_models,
    quantile=None,
    cv=2,
    random_forest__n_estimators=80,
    extra_forest__n_estimators=80,
)
pipe.fit(train, valid_data=valid)
pipe.leader_board_

## Direct model classes / 直接模型类

For single-model experiments you can instantiate wrapper classes directly. In production tutorials we recommend registry names through `ModelPipeline`, because they work with `PipelineConfigs`, error resilience, logging, save/load, and `SmartRouter`.

对于单模型实验，可以直接实例化包装类。生产教程更推荐通过 `ModelPipeline` 使用注册名，因为它能统一支持 `PipelineConfigs`、错误容忍、日志、保存/加载和 `SmartRouter`。

In [ ]:
from PipelineTS.ml_model import RandomForestModel

model = RandomForestModel(
    time_col="date",
    target_col="sales",
    lags=12,
    quantile=0.9,
    n_estimators=120,
    random_state=42,
)
model.fit(train, cv=2)
model.predict(12).head()

In [ ]:
nn_candidates = [m for m in ["d_linear", "n_linear", "tide", "n_hits", "itransformer", "srs_net"] if m in available]
optional_candidates = [m for m in ["chronos_2", "chronos_2_synth", "chronos_2_small"] if m in available]

print("NN candidates available:", nn_candidates)
print("Optional foundation models available:", optional_candidates)